## Base Model training

We start by training and testing VGG16 and Densenet121 on the task. <br>
We then go on to evaluate the performance MMSEN on the task.

In [1]:
import sys
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = "/content/drive/MyDrive/Projects/MMSEN"
sys.path.insert(0, PROJECT_ROOT)

Mounted at /content/drive


In [23]:
from torchvision import models, transforms
import torch
from torch import nn

from src.train import train_loop

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [14]:
mean = [0.7008, 0.5384, 0.6916]
std = [0.2350, 0.2774, 0.2129]

In [17]:
# define a transform, now normalize using the computed mean and std,
# also add random horizontal and vertical flipping for dataset augmentation

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

### VGG16

In [5]:
# Load VGG16 with pretrained ImageNet weights
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 190MB/s] 


In [6]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [7]:
class VGG16_GAP_Classifier(nn.Module):
  
  def __init__(self):
    super().__init__()
    self.features = vgg16.features
    self.classifier = nn.Linear(512, 1)

  def forward(self, x):
    x = self.features(x)
    x = torch.mean(x, dim=(2, 3))
    x = self.classifier(x)
    return x

In [8]:
vgg16 = VGG16_GAP_Classifier()

In [9]:
### pass the model to the correct device
vgg16 = vgg16.to(device, memory_format=torch.channels_last)

### freeze all layers apart from the classifier

for param in vgg16.parameters():
  param.requires_grad = False

for param in vgg16.classifier.parameters():
  param.requires_grad = True

In [24]:
vgg16, train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc, classes_per_epoch = train_loop(
    vgg16, 
    "vgg16_classifier_10epochs",
    train_transform,
    test_transform
)

TypeError: load_data() missing 2 required positional arguments: 'train_transform' and 'test_transform'